In [ ]:
!pip install yfinance --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
train_df = yf.download('GOOGL', start='2012-01-01', end='2017-01-01', auto_adjust=True).reset_index()
test_df  = yf.download('GOOGL', start='2017-01-01', end='2017-02-01', auto_adjust=True).reset_index()
print('Train:', train_df.shape, '| Test:', test_df.shape)
print(train_df[['Date','Open','High','Low','Close','Volume']].head())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train_df['Date'], train_df['Open'], color='blue')
plt.title('Google Stock Open Price (2012-2016)')
plt.xlabel('Date'); plt.ylabel('Price (USD)')
plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
training_set = train_df[['Open']].values
scaler = MinMaxScaler(feature_range=(0,1))
scaled = scaler.fit_transform(training_set)

TIMESTEPS = 60
X_train, y_train = [], []
for i in range(TIMESTEPS, len(scaled)):
    X_train.append(scaled[i-TIMESTEPS:i, 0])
    y_train.append(scaled[i, 0])

X_train = np.array(X_train).reshape(-1, TIMESTEPS, 1)
y_train = np.array(y_train)
print('X_train:', X_train.shape, '| y_train:', y_train.shape)

In [ ]:
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(TIMESTEPS,1)), Dropout(0.2),
    LSTM(50, return_sequences=True), Dropout(0.2),
    LSTM(50, return_sequences=True), Dropout(0.2),
    LSTM(50), Dropout(0.2),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], color='blue', label='Train Loss')
plt.title('Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(True); plt.show()

In [ ]:
actual = test_df[['Open']].values
total  = pd.concat((train_df[['Open']], test_df[['Open']]), axis=0)
inputs = scaler.transform(total[len(total)-len(test_df)-TIMESTEPS:].values.reshape(-1,1))

X_test = []
for i in range(TIMESTEPS, TIMESTEPS + len(test_df)):
    X_test.append(inputs[i-TIMESTEPS:i, 0])
X_test = np.array(X_test).reshape(-1, TIMESTEPS, 1)

predicted = scaler.inverse_transform(model.predict(X_test))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(actual,    color='red',  label='Actual Price',    linewidth=2)
plt.plot(predicted, color='blue', label='Predicted Price', linewidth=2, linestyle='--')
plt.title('Google Stock Price Prediction – January 2017')
plt.xlabel('Trading Day'); plt.ylabel('Price (USD)')
plt.legend(); plt.grid(True); plt.show()

In [ ]:
rmse = np.sqrt(mean_squared_error(actual, predicted))
mae  = mean_absolute_error(actual, predicted)
mape = np.mean(np.abs((actual - predicted) / actual)) * 100
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'MAPE : {mape:.2f}%')